In [4]:
#合并csv数据文件
import pandas as pd
import os
Folder_Path = r'E:\UAV\only_comments\cleaned'          #要拼接的文件夹及其完整路径，注意不要包含中文
SaveFile_Path =  r'E:\UAV'       #拼接后要保存的文件路径
SaveFile_Name = r'all_UAV_data.csv'              #合并后要保存的文件名
 
#修改当前工作目录
os.chdir(Folder_Path)
#将该文件夹下的所有文件名存入一个列表
file_list = os.listdir()
 
#读取第一个CSV文件并包含表头
df = pd.read_csv(Folder_Path +'\\'+ file_list[0])   #编码默认UTF-8，若乱码自行更改
 
#将读取的第一个CSV文件写入合并后的文件保存
df.to_csv(SaveFile_Path+'\\'+ SaveFile_Name,encoding="utf_8_sig",index=False)
 
#循环遍历列表中各个CSV文件名，并追加到合并后的文件
for i in range(1,len(file_list)):
    df = pd.read_csv(Folder_Path + '\\'+ file_list[i])
    df.to_csv(SaveFile_Path+'\\'+ SaveFile_Name,encoding="utf_8_sig",index=False, header=False, mode='a+')

In [10]:
#对文本评论进行分词
import jieba
import pandas as pd
import re

# 加载停用词库
def load_stopwords(path):
    with open(path, 'r', encoding='utf-8') as f:
        stopwords = [line.strip() for line in f.readlines()]
    return set(stopwords)

# 分词并去除停用词
def segment_and_remove(text, stopwords):
    words = jieba.cut(text)
    return ' '.join([word for word in words if word not in stopwords])

def filter_emoji(desstr, restr=''):
    # 过滤表情
    try:
        co = re.compile(u'[\U00010000-\U0010ffff]')
    except re.error:
        co = re.compile(u'[\uD800-\uDBFF][\uDC00-\uDFFF]')
    return co.sub(restr, desstr)

# 读取CSV文件
df = pd.read_csv(r'E:\UAV\all_UAV_data.csv', encoding='utf-8')
# 对评论内容进行emoji过滤
df['评论内容'] = df['评论内容'].apply(filter_emoji)

# 加载停用词
stopwords = load_stopwords(r'E:\UAV\stopwords\hit_stopwords.txt')

# 对评论内容进行分词和去除停用词
column_data = df['评论内容'].apply(lambda x: segment_and_remove(x, stopwords))

# 保存清洗后的数据到新的CSV文件
column_data.to_csv(r'E:\UAV\cut_UAV_data.csv', encoding='utf_8_sig', index=False)

print(column_data)

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\Godfather\AppData\Local\Temp\jieba.cache
Loading model cost 0.533 seconds.
Prefix dict has been built successfully.


0                             亲测 这款 飞机 很耐炸
1                                    松鼠 老师
2                      款 适合 新手 玩 带 遥控器 六千多
3                                     飞 不错
4                                老师 真的 太 牛
                       ...                
31722                                   型号
31723             信号 丢失 没有 返航 大疆 说 飞到 水 里面
31724                                设置 问题
31725                                电池 自由
31726    今晚 突然 断 掉 别人 家 阳台 上 还 不 在家 现在 还 躺
Name: 评论内容, Length: 31727, dtype: object


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 步骤1: 读取CSV文件
df = pd.read_csv(r'E:\UAV\cut_UAV_data.csv', encoding='utf-8')
# 假设CSV文件中包含一个名为'column_name'的列，存放文本数据
text_data = df['评论内容']

# 步骤2: 移除包含NaN值的行
df = df.dropna(subset=['评论内容'])

# 步骤3: 确保所有文本数据都是字符串类型
text_data = df['评论内容'].astype(str)

# 步骤4: 使用TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(text_data)

# 步骤5: 分析结果
feature_names = vectorizer.get_feature_names_out()
tfidf_text = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# 打印整个数据集的每个词的总TF-IDF值
total_tfidf = tfidf_text.sum(axis=0)

Words_Totaltfidf = pd.DataFrame({
    '词语': total_tfidf.index,
    '总TF-IDF值': total_tfidf.values})

print("每个词的总体TF-IDF值:")
print(total_tfidf)

# 提取并打印整个数据集的关键词
N = 500  # 你可以根据需要调整这个数字
keywords = total_tfidf.nlargest(N).index.tolist()
print("数据集当中的top关键词")
for word in keywords:
    print(f"{word}: {total_tfidf[word]}")

# 如果你想查看整个DataFrame
# print(tfidf_df.head())

# 可选：保存TF-IDF DataFrame到CSV文件
Words_Totaltfidf.to_csv(r'E:\UAV\UAV_Words_Totaltfidf.csv', encoding='utf_8_sig', index=False)

每个词的总体TF-IDF值:
00      5.976988
000     1.198771
0029    0.413094
007     0.433437
01      2.915044
          ...   
龙头老大    0.646927
龙年      0.328339
龙舟      0.350399
龙门石窟    0.356385
龟速      0.202139
Length: 24477, dtype: float64
数据集当中的top关键词
非常: 772.598543249651
无人机: 692.4814527705165
大疆: 672.1791663781815
不错: 641.3796195057865
操作: 547.2678230118945
质量: 409.3918425623802
没有: 353.48719210775107
简单: 320.87525927622505
问题: 314.66518366567
飞机: 313.28720751223057
做工: 310.55259077011294
喜欢: 310.32945811530277
电池: 294.7675427887659
飞行: 293.4525117286828
稳定性: 284.72641387968036
真的: 272.36594848407606
感觉: 272.022658327332
新手: 268.89018087351405
值得: 258.50257054682424
东西: 257.04845771631597
视频: 253.35210617393284
很快: 250.34677873437377
灵敏度: 239.25633596012324
一下: 237.8785528503065
满意: 232.00082504045852
价格: 226.08389915684063
不是: 225.3451984235814
难易: 221.3601617150813
容易: 217.28482039040028
收到: 217.1162091897913
清晰: 214.13597622672566
方便: 213.38359785335945
物流: 210.87253466513906
产品: 202.353

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 步骤1: 读取CSV文件
df = pd.read_csv(r'E:\UAV\cut_UAV_data.csv', encoding='utf-8')
# 假设CSV文件中包含一个名为'评论内容'的列，存放文本数据
text_data = df['评论内容']

# 步骤2: 移除包含NaN值的行
df = df.dropna(subset=['评论内容'])

# 步骤3: 确保所有文本数据都是字符串类型
text_data = df['评论内容'].astype(str)

# 步骤4: 使用TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(text_data)

# 步骤5: 分析结果
feature_names = vectorizer.get_feature_names_out()
tfidf_text = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

# 打印整个数据集的每个词的平均TF-IDF值
# 使用mean(axis=0)计算每个词的平均TF-IDF值
average_tfidf = tfidf_text.mean(axis=0)

Words_Averagetfidf = pd.DataFrame({
    '词语': average_tfidf.index,
    '平均TF-IDF值': average_tfidf.values})

print("每个词的平均TF-IDF值:")
print(average_tfidf)

# 提取并打印整个数据集的关键词
N = 500  # 你可以根据需要调整这个数字
keywords = average_tfidf.nlargest(N).index.tolist()
print("数据集当中的top关键词")
for word in keywords:
    print(f"{word}: {average_tfidf[word]}")

# 如果你想查看整个DataFrame
# print(tfidf_df.head())

# 可选：保存平均TF-IDF DataFrame到CSV文件
Words_Averagetfidf.to_csv(r'E:\UAV\UAV_Words_Averagetfidf.csv', encoding='utf_8_sig', index=False)

每个词的平均TF-IDF值:
00      0.000189
000     0.000038
0029    0.000013
007     0.000014
01      0.000092
          ...   
龙头老大    0.000020
龙年      0.000010
龙舟      0.000011
龙门石窟    0.000011
龟速      0.000006
Length: 24477, dtype: float64
数据集当中的top关键词
非常: 0.024391429936847703
无人机: 0.02186208217112917
大疆: 0.021221126010360903
不错: 0.020248764625281343
操作: 0.017277595043785145
质量: 0.012924762196128814
没有: 0.011159816641128684
简单: 0.010130237072651146
问题: 0.009934181015490765
飞机: 0.009890677427379023
做工: 0.009804343828574993
喜欢: 0.009797299388012716
电池: 0.00930599977233673
飞行: 0.009264483401063388
稳定性: 0.008988994913328503
真的: 0.008598767118676433
感觉: 0.008587929228960758
新手: 0.00848903491313383
值得: 0.00816109141426438
东西: 0.008115184142582983
视频: 0.007998487961292275
很快: 0.007903607852703198
灵敏度: 0.007553475484139645
一下: 0.007509977990538485
满意: 0.007324414365918185
价格: 0.007137613233049428
不是: 0.007114291978645032
难易: 0.006988481822102014
容易: 0.006859820691093932
收到: 0.006854497527696647
清晰: 0.